# First Question
To get further aquainted with the data and its possibilitys I want to try answering one of the example questions provided. It seemd to me that the first one: "Which neighbourhoods receive the highest number of reports?" is straigt forward and is an easy beginning. This Notebook is dedicated to answer this question and maybe get an idea for additional questions I could answer. It was also recommended to start with a clear highly focused question.

### Workflow
1. Identify data and attributes needed
2. Make plan/vision what coding steps are needed
3. Code

### 1. Identify data and Attributes needed
Both datasets, ZuriWieNeu and Quartiere, are needed. ZuriWieNeu for the Number of reports and Quartiere to distribute these numbers to the coresponding neigbourhood via spatial relationships. Therefor the following attributes are used:

For Quartiere (Layer = ADM_STATISTISCHE_QUARTIERE_V):
| Name | Definition |
|------|------------|
| GEOMETRIE | Geometriefeld |
| NAME | Bezeichnung des statistischen Quartiers |


For ZuriWieNeu
| Name | Definition |
|------|------------|
| GEOMETRIE | Geometriefeld. Typ SHAPE. |


### 2. Plan/Vision
1. Load Librarys
2. Load Data
3. Filter data
4. Use spatial relationships to distribute points on neighbourhoods
5. Count Reports per neighbourhood
6. Get neighbourhood with highest count
7. Answer question

### 3. Code
(Preparations for loading data were made in "Project_Start.ipynb")

In [24]:
# Load Librarys
import pandas as pd
import geopandas as gpd 
import numpy as np

# Load data
data_gpkg = gpd.read_file("../data/raw/ZuriWieNeu_gpkg_data/data.gpkg").to_crs(epsg=2056)
boundaries_gpkg_v = gpd.read_file("../data/raw/boundaries_GPKG/data.gpkg",
 layer="stzh.adm_statistische_quartiere_v").to_crs(epsg=2056) # Layer for mapping



In [25]:
# Filter data
data_gpkg.columns.values
boundaries_gpkg_v.columns.values
data_subset = data_gpkg[["geometry"]]
boundaries_subset = boundaries_gpkg_v[["geometry", "qname"]]
display(data_subset.head(2))
display(boundaries_subset.head(6))

,geometry
0,POINT (2678968 1247548)
1,POINT (2680746 1249916)


,geometry,qname
0,"POLYGON ((2680606.662 1247034.584, 2680626.356...",Alt-Wiedikon
1,"POLYGON ((2685858.632 1246502.629, 2685860.738...",Witikon
2,"POLYGON ((2681313.304 1248613.857, 2681459.605...",Langstrasse
3,"POLYGON ((2680009.144 1249565.021, 2680055.843...",Escher Wyss
4,"POLYGON ((2681898.171 1246379.668, 2681899.115...",Enge
5,"POLYGON ((2684268.476 1246568.755, 2684268.988...",Weinegg


In [26]:
# Code to get numbers of reports per neighbourhood


# Spatial Join: distribute points to neighbourhoods
joined = gpd.sjoin(data_subset, boundaries_subset, predicate="within")

# count points per neighbourhood
counts = joined.groupby(joined.index_right).size()

# save count in boundaries_subset
boundaries_subset["n_reports"] = counts

# view attribute table
display(boundaries_subset[["qname", "n_reports"]].head(3))


,qname,n_reports
0,Alt-Wiedikon,2524
1,Witikon,1141
2,Langstrasse,6214


In [27]:
# Code to get neighbourhood with highes count of reports

max_count = boundaries_subset["n_reports"].idxmax()

# Get name of neighbourhood and count as variables
neighbourhood = boundaries_subset.loc[max_count, "qname"]
report_count = boundaries_subset.loc[max_count, "n_reports"]

# print Answer to question
print(
    f"The answer to the question:\n"
    f"Which neighbourhoods receive the highest number of reports?\n"
    f"is: {neighbourhood} with {report_count} Reports")

The answer to the question:
Which neighbourhoods receive the highest number of reports?
is: Langstrasse with 6214 Reports
